In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
OUT_DIR = Path("analysis_outputs")
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
def summarize_df(df: pd.DataFrame, name: str) -> dict:
    info = {}
    info["name"] = name
    info["rows"] = int(df.shape[0])
    info["cols"] = int(df.shape[1])
    info["memory_usage_bytes"] = int(df.memory_usage(deep=True).sum())

    dtypes = df.dtypes.apply(lambda x: x.name).to_dict()
    info["dtypes"] = dtypes

    # Missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)).round(4).to_dict()
    info["missing_count"] = missing.to_dict()
    info["missing_pct"] = missing_pct

    # Basic stats
    numeric = df.select_dtypes(include=[np.number])
    categorical = df.select_dtypes(exclude=[np.number])

    info["numeric_count"] = int(numeric.shape[1])
    info["categorical_count"] = int(categorical.shape[1])

    if not numeric.empty:
        desc = numeric.describe().to_dict()
        info["numeric_describe"] = desc
        # correlations
        corr = numeric.corr()
        info["numeric_corr_top_abs_pairs"] = (
            corr.abs().stack()
            .reset_index()
            .query('level_0 != level_1')
            .rename(columns={0: 'abs_corr', 'level_0':'f1','level_1':'f2'})
            .sort_values('abs_corr', ascending=False)
            .head(20)
            .to_dict(orient='records')
        )

    # Unique values and top values
    uniques = {col: int(df[col].nunique(dropna=False)) for col in df.columns}
    top_values = {}
    for col in df.columns:
        top = df[col].value_counts(dropna=False).head(5).to_dict()
        top_values[col] = top
    info["unique_counts"] = uniques
    info["top_values"] = top_values

    # Duplicates
    info["duplicates"] = int(df.duplicated().sum())

    # Try detect datetime columns
    datetime_cols = []
    for col in df.columns:
        if df[col].dtype == 'object':
            try:
                parsed = pd.to_datetime(df[col], errors='coerce')
                if parsed.notnull().sum() / max(1, len(parsed)) > 0.5:
                    datetime_cols.append(col)
            except Exception:
                pass
        elif np.issubdtype(df[col].dtype, np.datetime64):
            datetime_cols.append(col)

    info["datetime_like_columns"] = datetime_cols

    return info


In [ ]:

def save_json(obj, out_path: Path):
    with out_path.open('w', encoding='utf8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [ ]:
def plot_numeric_distributions(df: pd.DataFrame, prefix: str):
    numeric = df.select_dtypes(include=[np.number])
    if numeric.shape[1] == 0:
        return
    # Histograms
    for col in numeric.columns:
        plt.figure(figsize=(6, 3))
        sns.histplot(numeric[col].dropna(), kde=True, bins=50)
        plt.title(f"{prefix} - {col}")
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{prefix}_{col}_hist.png")
        plt.close()

    # Correlation heatmap (top 30 features by variance)
    try:
        var = numeric.var().sort_values(ascending=False)
        top = var.head(30).index.tolist()
        corr = numeric[top].corr()
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr, cmap='RdBu_r', center=0)
        plt.title(f"{prefix} - top numeric corr")
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{prefix}_corr_heatmap.png")
        plt.close()
    except Exception:
        pass


In [ ]:
def inspect(paths: dict):
    results = {}
    for name, path in paths.items():
        p = Path(path)
        if not p.exists():
            print(f"Skipping missing file: {p}")
            continue
        print(f"Loading {name}: {p}")
        df = pd.read_csv(p)
        print(f"Summarizing {name} (rows={len(df)}, cols={df.shape[1]})")
        info = summarize_df(df, name)
        results[name] = info
        save_json(info, OUT_DIR / f"{name}_summary.json")
        # save a small sample
        df.head(200).to_csv(OUT_DIR / f"{name}_sample.csv", index=False)
        plot_numeric_distributions(df, name)

    # Combined summary
    save_json(results, OUT_DIR / "combined_summary.json")
    print("Analysis completed. Outputs in:", OUT_DIR)



In [ ]:
def main():
    base = Path("dataset")
    paths = {
        "train": base / "train.csv",
        "test": base / "test.csv",
        "sample_submission": base / "sample_submission.csv",
    }
    inspect(paths)


if __name__ == '__main__':
    main()
